In [2]:
# Import and initialize the Earth Engine library.
import ee
import contextily
import geemap
from colorama import Fore, Back, Style

#my helper here
from scripts.kml_utils import parse_kml_coordinates

# (1)
# Authenticate and initialize Google Earth Engine
# (You must run ee.Authenticate() and ee.Initialize() beforehand)
# Its going to ask you to visit a URL, sign in with your Google account,
# and you most login with the google account associated to your google earth engine Project.
ee.Authenticate()
print(Fore.RED + 'Authentication complete' + Fore.RESET)

# (2)
# Connect to a specific GEE project
# In my case, its the account associated to the project "eastern-thinker-471320-h4"
ee.Initialize(project='eastern-thinker-471320-h4')
print(Back.GREEN + 'Conection to project established' + Back.RESET)

# (3)
# Test the connection by printing a message from the Earth Engine servers.
print(Fore.CYAN + ee.String('Ping from Google Earth Engine!').getInfo() + Fore.RESET)

# (4)
# Lets look for the GEDI image collections at different spatial resolutions
ic_1k = ee.ImageCollection('LARSE/GEDI/GRIDDEDVEG_002/V1/1KM')
dem = ic_1k.geometry
dem = ic_1k.select('median').first()
ic_6k = ee.ImageCollection('LARSE/GEDI/GRIDDEDVEG_002/V1/6KM')
dem_6k = ic_6k.select('median').first()
ic_12k = ee.ImageCollection('LARSE/GEDI/GRIDDEDVEG_002/V1/12KM')
dem_12k = ic_12k.select('median').first()

# (5)
# comparing the three DEMs from SRTM and GEDI
srtm = ee.Image('USGS/SRTMGL1_003')
dem = srtm
print(Fore.YELLOW + 'SRTM DEM info:' + Fore.RESET)
print(dem.getInfo())
print(Fore.YELLOW + 'GEDI 6K DEM info:' + Fore.RESET)
print(dem_6k.getInfo())
print(Fore.YELLOW + 'GEDI 12K DEM info:' + Fore.RESET)
print(dem_12k.getInfo())    


# Set visualization parameters
vis_params = {
    'min': 0,
    'max': 4000,
    'palette': ['006633', 'E5FFCC', '662A00', 'D8D8D8', 'F5F5F5']
}

# (6)
# Our region of interest (ROI) is Black Hills, South Dakota
#blackHills = ee.Geometry.Polygon( [[[-104.2, 44.9], [-104.2, 43.0], [-103.0, 43.0], [-103.0, 44.9], [-104.2, 44.9]]] )
# Create a map centered on the ROI
Map = geemap.Map()

# (7)
# Parse the KML file to get the coordinates and create a geometry
# BlackHills Polygon
# uncomment (original polygon)
blackHills_coords = parse_kml_coordinates('polygons/BlackHills.kml')
# BlackHills_reduced
# blackHills_coords = parse_kml_coordinates('BlackHills_reduced.kml')
# BlackHills_reduced-smoothed
# blackHills_coords = parse_kml_coordinates('BlackHills_reduced_smoothed.kml')

# BlackHills Geometry
blackHills_geom = ee.Geometry.Polygon([blackHills_coords])

#coords = geoJSON['features'][0]['geometry']['coordinates']
blackHills_geom = ee.Geometry.Polygon(blackHills_coords)
Map.addLayer(blackHills_geom, {}, 'Black Hills Area')

# (8)
# Add the DEM layers to the map with different opacities
Map.addLayer(dem.clip(blackHills_geom), vis_params, 'SRTM DEM', opacity=0.5)
Map.addLayer(dem_6k.clip(blackHills_geom), vis_params, 'GEDI 6K DEM', opacity=0.25)
Map.addLayer(dem_12k.clip(blackHills_geom), vis_params, 'GEDI 12K DEM', opacity=0.1)
Map.center_object(blackHills_geom, zoom=8)
Map


Authentication complete
Conection to project established
Ping from Google Earth Engine!
SRTM DEM info:
{'type': 'Image', 'bands': [{'id': 'elevation', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': -32768, 'max': 32767}, 'dimensions': [1296001, 417601], 'crs': 'EPSG:4326', 'crs_transform': [0.0002777777777777778, 0, -180.0001388888889, 0, -0.0002777777777777778, 60.00013888888889]}], 'version': 1641990767055141, 'id': 'USGS/SRTMGL1_003', 'properties': {'system:visualization_0_min': '0.0', 'type_name': 'Image', 'keywords': ['dem', 'elevation', 'geophysical', 'nasa', 'srtm', 'topography', 'usgs'], 'thumb': 'https://mw1.google.com/ges/dd/images/SRTM90_V4_thumb.png', 'description': '<p>The Shuttle Radar Topography Mission (SRTM, see <a href="https://onlinelibrary.wiley.com/doi/10.1029/2005RG000183/full">Farr\net al. 2007</a>)\ndigital elevation data is an international research effort that\nobtained digital elevation models on a near-global scale. This\nSRTM V3 product (SRTM

Map(center=[43.988020092438674, -103.73714829146317], controls=(WidgetControl(options=['position', 'transparen…